In [4]:
# ============================================================
# MODEL 4 — iTransformer  (Local PC version)
# GPU: NVIDIA RTX 4070 Super (12 GB VRAM)
#
# Run from terminal:
#   python Train_model4_itransformer_local.py
#
# Requirements (install once):
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
#   pip install pandas numpy scikit-learn
#
# Files needed in C:\Users\user1\Downloads\iTransformer\
#   X_train_enriched.csv
#   X_test_enriched.csv
#   y_train.csv
#   y_test.csv
#
# Outputs saved to:
#   C:\Users\user1\Downloads\iTransformer\model_outputs\
# ============================================================

import os
import time
import json
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("  WARNING: No GPU found. Training will be slow on CPU.")
    print("  Make sure PyTorch CUDA is installed correctly.")

# ============================================================
# PATHS  (same as your previous local run)
# ============================================================

DRIVE_DIR  = r'C:\Users\user1\Downloads\iTransformer'
OUTPUT_DIR = os.path.join(DRIVE_DIR, 'model_outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROGRESS_FILE   = os.path.join(OUTPUT_DIR, 'model4_search_progress.json')
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'model4_checkpoint.pt')

print(f"\nData directory   : {DRIVE_DIR}")
print(f"Output directory : {OUTPUT_DIR}")

# ============================================================
# CONSTANTS
# ============================================================

TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]
ID_COLS     = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']
CLIP_RANGES = [(2,8), (5,200), (1,7), (1,7)]
QUANTILES   = [0.10, 0.50, 0.90]

# ============================================================
# SETTINGS
# ============================================================

N_SEARCH_ITER     = 8
VAL_FRACTION      = 0.20
MAX_EPOCHS_SEARCH = 35
MAX_EPOCHS_FINAL  = 60
PATIENCE_SEARCH   = 4
PATIENCE_FINAL    = 10
# RTX 4070 Super has 12 GB VRAM — 2048 is safe for this model size
BATCH_SIZE        = 2048
GRAD_CLIP         = 1.0
RANDOM_SEED       = 42
CKPT_EVERY        = 5
# Windows: num_workers > 0 requires if __name__ == '__main__' guard
# Setting to 0 avoids multiprocessing issues on Windows
NUM_WORKERS       = 0

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ============================================================
# SEARCH SPACE
# ============================================================

SEARCH_SPACE = {
    'd_model_heads': [
        (64,  4),
        (64,  8),
        (128, 4),
        (128, 8),
    ],
    'n_layers':    [2, 4],
    'd_ff_factor': [2, 4],
    'dropout':     [0.05, 0.1, 0.2],
    'lr':          [1e-4, 5e-4, 1e-3],
    'weight_decay':[1e-5, 1e-4],
}

def sample_config(seed):
    rng     = np.random.RandomState(seed)
    idx     = int(rng.randint(len(SEARCH_SPACE['d_model_heads'])))
    d_model, n_heads = SEARCH_SPACE['d_model_heads'][idx]
    d_ff_factor = int(rng.choice(SEARCH_SPACE['d_ff_factor']))
    return {
        'd_model':      int(d_model),
        'n_heads':      int(n_heads),
        'n_layers':     int(rng.choice(SEARCH_SPACE['n_layers'])),
        'd_ff':         int(d_model * d_ff_factor),
        'dropout':      float(rng.choice(SEARCH_SPACE['dropout'])),
        'lr':           float(rng.choice(SEARCH_SPACE['lr'])),
        'weight_decay': float(rng.choice(SEARCH_SPACE['weight_decay'])),
    }

# ============================================================
# LOAD DATA
# ============================================================

def load_data():
    print("\nLoading data...")

    # Check files exist before trying to load
    needed = [
        'X_train_enriched.csv',
        'X_test_enriched.csv',
        'y_train.csv',
        'y_test.csv',
    ]
    for fname in needed:
        path = os.path.join(DRIVE_DIR, fname)
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"\nFile not found: {path}\n"
                f"Make sure all four CSV files are in:\n  {DRIVE_DIR}"
            )

    X_train_df = pd.read_csv(
        os.path.join(DRIVE_DIR, 'X_train_enriched.csv'), low_memory=False
    )
    X_test_df  = pd.read_csv(
        os.path.join(DRIVE_DIR, 'X_test_enriched.csv'),  low_memory=False
    )
    y_train_df = pd.read_csv(os.path.join(DRIVE_DIR, 'y_train.csv'))
    y_test_df  = pd.read_csv(os.path.join(DRIVE_DIR, 'y_test.csv'))

    assert len(X_train_df) == 201984, \
        f"Expected 201984 train rows, got {len(X_train_df)}"
    assert len(X_test_df)  == 135120, \
        f"Expected 135120 test rows, got {len(X_test_df)}"

    print(f"  X_train : {X_train_df.shape}")
    print(f"  X_test  : {X_test_df.shape}")

    # Fix boolean columns
    for df in [X_train_df, X_test_df]:
        for col in df.columns:
            if col in ID_COLS:
                continue
            if df[col].dtype == bool:
                df[col] = df[col].astype(int)
            elif df[col].dtype == object:
                vals = set(df[col].dropna().astype(str).unique())
                if vals <= {'True', 'False'}:
                    df[col] = df[col].map({'True': 1, 'False': 0}).astype(int)

    feat_cols = [c for c in X_train_df.columns if c not in ID_COLS]

    # Impute missing values with training median
    for col in feat_cols:
        if X_train_df[col].isnull().any():
            med = X_train_df[col].median()
            X_train_df[col] = X_train_df[col].fillna(med)
            X_test_df[col]  = X_test_df[col].fillna(med)

    print(f"  Features: {len(feat_cols)}")
    print("  Dataset loaded successfully.")
    return X_train_df, X_test_df, y_train_df, y_test_df, feat_cols


# ============================================================
# DATASET
# ============================================================

class iTransformerDataset(Dataset):
    def __init__(self, X_df, y_df, feat_cols):
        self.X = torch.tensor(
            X_df[feat_cols].values.astype(np.float32)
        )
        self.y = torch.tensor(
            y_df[TARGET_COLS].values.astype(np.float32)
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_loaders(X_df, y_df, feat_cols,
                 X_val_df=None, y_val_df=None):
    tr_dl = DataLoader(
        iTransformerDataset(X_df, y_df, feat_cols),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    )
    if X_val_df is not None:
        val_dl = DataLoader(
            iTransformerDataset(X_val_df, y_val_df, feat_cols),
            batch_size=BATCH_SIZE * 2, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
        )
        return tr_dl, val_dl
    return tr_dl

# ============================================================
# MODEL
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class iTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        r = x; x = self.norm1(x)
        x, _ = self.attn(x, x, x)
        x = self.drop(x) + r
        r = x; x = self.norm2(x)
        x = self.ff(x) + r
        return x


class iTransformerModel(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8,
                 n_layers=4, d_ff=512, dropout=0.1,
                 output_dim=4, n_quantiles=3):
        super().__init__()
        self.feature_embedding = nn.Linear(1, d_model)
        self.feature_pos_enc   = nn.Parameter(
            torch.randn(1, n_features, d_model) * 0.02
        )
        self.encoder = nn.ModuleList([
            iTransformerLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model)
        self.pre_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.LayerNorm(d_model // 2),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.quantile_heads = nn.ModuleList([
            nn.Linear(d_model // 2, output_dim)
            for _ in range(n_quantiles)
        ])
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.feature_embedding(x.unsqueeze(-1)) + self.feature_pos_enc
        for layer in self.encoder:
            x = layer(x)
        h = self.pre_head(self.encoder_norm(x).mean(dim=1))
        return [head(h) for head in self.quantile_heads]


class MultiQuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, preds, y_true):
        total = 0.0
        for q, pred in zip(self.quantiles, preds):
            diff  = y_true - pred
            total += torch.where(
                diff >= 0, q * diff, (q - 1) * diff
            ).mean()
        return total / len(self.quantiles)

# ============================================================
# HELPERS
# ============================================================

def count_params(config, n_features):
    m = iTransformerModel(
        n_features=n_features, **{
            k: config[k] for k in
            ['d_model','n_heads','n_layers','d_ff','dropout']
        }, output_dim=4, n_quantiles=len(QUANTILES)
    )
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    del m; return n


def build_model_and_opt(config, n_features, n_steps, max_epochs):
    model = iTransformerModel(
        n_features=n_features, **{
            k: config[k] for k in
            ['d_model','n_heads','n_layers','d_ff','dropout']
        }, output_dim=4, n_quantiles=len(QUANTILES)
    ).to(DEVICE)
    criterion = MultiQuantileLoss(QUANTILES)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['lr'], weight_decay=config['weight_decay']
    )
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=config['lr'],
        epochs=max_epochs, steps_per_epoch=n_steps,
        pct_start=0.1, anneal_strategy='cos',
    )
    return model, criterion, optimizer, scheduler


def run_epoch(model, loader, criterion,
              optimizer=None, scheduler=None):
    if optimizer:
        model.train()
    else:
        model.eval()
    total = 0.0
    ctx   = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            preds = model(X)
            loss  = criterion(preds, y)
            if optimizer:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                if scheduler:
                    scheduler.step()
            total += loss.item()
    return total / len(loader)


def evaluate(model, X_df, y_df, feat_cols):
    dl = DataLoader(
        iTransformerDataset(X_df, y_df, feat_cols),
        batch_size=BATCH_SIZE * 2, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'),
    )
    model.eval()
    q10, q50, q90 = [], [], []
    with torch.no_grad():
        for X, _ in dl:
            preds = model(X.to(DEVICE))
            q10.append(preds[0].cpu().numpy())
            q50.append(preds[1].cpu().numpy())
            q90.append(preds[2].cpu().numpy())

    y_q10 = np.vstack(q10)
    y_q50 = np.vstack(q50)
    y_q90 = np.vstack(q90)
    y_pred = y_q90.copy()
    for i, (lo, hi) in enumerate(CLIP_RANGES):
        y_pred[:, i] = np.clip(y_pred[:, i], lo, hi)

    y_true     = y_df[TARGET_COLS].values.astype(np.float32)
    per_target = {}
    all_mae, all_rmse, all_r2 = [], [], []
    for i, col in enumerate(TARGET_COLS):
        mae  = mean_absolute_error(y_true[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
        r2   = r2_score(y_true[:, i], y_pred[:, i])
        per_target[col] = {
            'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)
        }
        all_mae.append(mae); all_rmse.append(rmse); all_r2.append(r2)

    return (y_pred, y_q10, y_q50, y_q90, per_target,
            np.mean(all_mae), np.mean(all_rmse), np.mean(all_r2))

# ============================================================
# MAIN
# ============================================================

# Windows requires this guard when using multiprocessing
if __name__ == '__main__':

    # ── Load data ─────────────────────────────────────────────
    X_train_df, X_test_df, y_train_df, y_test_df, FEAT_COLS = load_data()
    N_FEATURES = len(FEAT_COLS)

    # ── Validation split (time-ordered) ───────────────────────
    print(f"\nCreating time-ordered validation split "
          f"({int(VAL_FRACTION*100)}%)...")
    n_val  = int(len(X_train_df) * VAL_FRACTION)
    n_tr   = len(X_train_df) - n_val

    X_tr_df  = X_train_df.iloc[:n_tr].reset_index(drop=True)
    X_val_df = X_train_df.iloc[n_tr:].reset_index(drop=True)
    y_tr_df  = y_train_df.iloc[:n_tr].reset_index(drop=True)
    y_val_df = y_train_df.iloc[n_tr:].reset_index(drop=True)

    print(f"  Train: {len(X_tr_df):,}   Val: {len(X_val_df):,}")

    # ── Load search progress (resume support) ─────────────────
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as f:
            saved = json.load(f)
        search_results = saved['completed']
        completed_ids  = {r['candidate'] for r in search_results}
        print(f"\nResuming search — "
              f"{len(completed_ids)} candidates already done: "
              f"{sorted(completed_ids)}")
    else:
        search_results = []
        completed_ids  = set()
        print("\nNo previous progress found — starting fresh.")

    # ── Hyperparameter search ──────────────────────────────────
    print(f"\n{'='*70}")
    print(f"HYPERPARAMETER SEARCH  ({N_SEARCH_ITER} candidates)")
    print(f"  Batch size    : {BATCH_SIZE}")
    print(f"  Search epochs : {MAX_EPOCHS_SEARCH}   "
          f"Patience : {PATIENCE_SEARCH}")
    print(f"{'='*70}")

    tr_loader, val_loader = make_loaders(
        X_tr_df, y_tr_df, FEAT_COLS, X_val_df, y_val_df
    )

    for i in range(N_SEARCH_ITER):

        cand_id = i + 1
        if cand_id in completed_ids:
            print(f"\nCandidate {cand_id:02d}/{N_SEARCH_ITER}  "
                  f"[already done — skipping]")
            continue

        config   = sample_config(seed=RANDOM_SEED + i)
        n_params = count_params(config, N_FEATURES)

        print(f"\nCandidate {cand_id:02d}/{N_SEARCH_ITER}")
        print(f"  d_model={config['d_model']}  "
              f"heads={config['n_heads']}  "
              f"layers={config['n_layers']}  "
              f"d_ff={config['d_ff']}  "
              f"drop={config['dropout']}  "
              f"lr={config['lr']:.0e}  "
              f"wd={config['weight_decay']:.0e}  "
              f"params={n_params:,}")

        t0 = time.time()

        try:
            model, criterion, optimizer, scheduler = build_model_and_opt(
                config, N_FEATURES, len(tr_loader), MAX_EPOCHS_SEARCH
            )
            best_val       = float('inf')
            patience_count = 0

            for epoch in range(1, MAX_EPOCHS_SEARCH + 1):
                run_epoch(model, tr_loader, criterion, optimizer, scheduler)
                val_loss = run_epoch(model, val_loader, criterion)

                if val_loss < best_val:
                    best_val       = val_loss
                    patience_count = 0
                else:
                    patience_count += 1
                    if patience_count >= PATIENCE_SEARCH:
                        break

            del model
            torch.cuda.empty_cache()

        except RuntimeError as e:
            print(f"  FAILED: {e}")
            best_val = float('inf')
            n_params = -1

        elapsed = time.time() - t0
        print(f"  Val loss : {best_val:.4f}  ({elapsed:.0f}s)")

        result = {
            'candidate': cand_id,
            'config':    config,
            'n_params':  n_params,
            'val_loss':  best_val,
            'time_sec':  round(elapsed, 1),
        }
        search_results.append(result)
        completed_ids.add(cand_id)

        # Save after every candidate
        with open(PROGRESS_FILE, 'w') as f:
            json.dump({'completed': search_results}, f, indent=2)
        print(f"  Progress saved → model4_search_progress.json")

    # ── Best configuration ────────────────────────────────────
    valid_results = [
        r for r in search_results if r['val_loss'] < float('inf')
    ]
    valid_results.sort(key=lambda x: x['val_loss'])
    best_result = valid_results[0]
    best_config = best_result['config']

    print(f"\n{'='*70}")
    print("SEARCH COMPLETE")
    print(f"{'='*70}")
    print(f"Best candidate : #{best_result['candidate']}")
    print(f"Best val loss  : {best_result['val_loss']:.4f}")
    print(f"Parameters     : {best_result['n_params']:,}")
    print("Best config    :")
    for k, v in best_config.items():
        print(f"  {k:<15} {v}")

    # ── Leaderboard ───────────────────────────────────────────
    print(f"\n{'='*70}")
    print("SEARCH LEADERBOARD")
    print(f"{'='*70}")
    print(f"{'Rank':>4} {'Cand':>4} {'Val Loss':>10} {'d_model':>8} "
          f"{'heads':>6} {'layers':>7} {'d_ff':>6} {'drop':>6} "
          f"{'lr':>8} {'Params':>10}")
    print("-" * 80)
    for rank, r in enumerate(valid_results, 1):
        c = r['config']
        print(f"  {rank:2d}     {r['candidate']:2d}   "
              f"{r['val_loss']:>10.4f}  "
              f"{c['d_model']:>7}  {c['n_heads']:>5}  "
              f"{c['n_layers']:>6}  {c['d_ff']:>5}  "
              f"{c['dropout']:>5.2f}  {c['lr']:>8.0e}  "
              f"{r['n_params']:>10,}")

    # ── Final model with checkpoint resume ────────────────────
    print(f"\n{'='*70}")
    print("FINAL MODEL — full training with best config + checkpointing")
    print(f"{'='*70}")

    full_tr_loader, full_val_loader = make_loaders(
        X_train_df, y_train_df, FEAT_COLS, X_val_df, y_val_df
    )

    final_model, criterion, optimizer, scheduler = build_model_and_opt(
        best_config, N_FEATURES,
        len(full_tr_loader), MAX_EPOCHS_FINAL
    )

    start_epoch    = 1
    best_val_final = float('inf')
    best_state     = None
    train_losses   = []
    val_losses     = []

    if os.path.exists(CHECKPOINT_FILE):
        print(f"\nCheckpoint found — loading {CHECKPOINT_FILE}")
        ckpt = torch.load(CHECKPOINT_FILE, map_location=DEVICE)
        if ckpt.get('config') == best_config:
            final_model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            start_epoch    = ckpt['epoch'] + 1
            best_val_final = ckpt['best_val_loss']
            best_state     = ckpt['best_state']
            train_losses   = ckpt.get('train_losses', [])
            val_losses     = ckpt.get('val_losses', [])
            print(f"  Resuming from epoch {start_epoch}  "
                  f"(best val so far: {best_val_final:.4f})")
        else:
            print("  Config mismatch — starting fresh.")

    if best_state is None:
        best_state = copy.deepcopy(final_model.state_dict())

    t_final        = time.time()
    patience_count = 0

    print(f"\n{'Epoch':>6} {'Train':>10} {'Val':>10} {'LR':>12}")
    print("-" * 45)

    for epoch in range(start_epoch, MAX_EPOCHS_FINAL + 1):

        tr_loss  = run_epoch(
            final_model, full_tr_loader, criterion, optimizer, scheduler
        )
        val_loss = run_epoch(final_model, full_val_loader, criterion)

        train_losses.append(tr_loss)
        val_losses.append(val_loss)

        lr_now = optimizer.param_groups[0]['lr']
        if epoch % 5 == 0 or epoch == 1:
            print(f"  {epoch:4d}   {tr_loss:>10.4f}   "
                  f"{val_loss:>8.4f}   {lr_now:>10.2e}")

        if val_loss < best_val_final:
            best_val_final = val_loss
            patience_count = 0
            best_state     = copy.deepcopy(final_model.state_dict())
        else:
            patience_count += 1
            if patience_count >= PATIENCE_FINAL:
                print(f"\n  Early stop at epoch {epoch}")
                break

        # Save checkpoint every CKPT_EVERY epochs
        if epoch % CKPT_EVERY == 0:
            torch.save({
                'epoch':               epoch,
                'config':              best_config,
                'model_state_dict':    final_model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(),
                'scheduler_state_dict':scheduler.state_dict(),
                'best_val_loss':       best_val_final,
                'best_state':          best_state,
                'train_losses':        train_losses,
                'val_losses':          val_losses,
            }, CHECKPOINT_FILE)
            print(f"  [Epoch {epoch}] Checkpoint saved")

    train_time   = time.time() - t_final
    total_params = sum(
        p.numel() for p in final_model.parameters()
        if p.requires_grad
    )
    final_model.load_state_dict(best_state)

    print(f"\nFinal training time : {train_time:.1f}s "
          f"({train_time/60:.1f} min)")
    print(f"Model parameters    : {total_params:,}")

    # ── Test evaluation ───────────────────────────────────────
    print(f"\n{'='*70}")
    print("TEST RESULTS  (threshold = P90 quantile output)")
    print(f"{'='*70}")

    (y_pred, y_q10, y_q50, y_q90,
     per_target, avg_mae, avg_rmse, avg_r2) = evaluate(
        final_model, X_test_df, y_test_df, FEAT_COLS
    )

    print(f"\n{'Target':<35} {'MAE':>8} {'RMSE':>8} {'R2':>8}")
    print("-" * 62)
    for col in TARGET_COLS:
        r = per_target[col]
        short = col.replace('TARGET_', '')
        print(f"  {short:<33} "
              f"{r['MAE']:>8.4f} {r['RMSE']:>8.4f} {r['R2']:>8.4f}")
    print("-" * 62)
    print(f"  {'AVERAGE':<33} "
          f"{avg_mae:>8.4f} {avg_rmse:>8.4f} {avg_r2:>8.4f}")

    # ── Quantile spread ───────────────────────────────────────
    print(f"\nQuantile spread (P10 to P90) per target:")
    for i, col in enumerate(TARGET_COLS):
        spread = np.mean(np.abs(y_q90[:, i] - y_q10[:, i]))
        short  = col.replace('TARGET_', '')
        unit   = "hours" if i == 0 else ("minutes" if i == 1 else "")
        print(f"  {short:<35} avg spread = {spread:.4f} {unit}")

    # ── Save outputs ──────────────────────────────────────────
    pd.DataFrame(
        y_pred, columns=[c+'_pred' for c in TARGET_COLS]
    ).to_csv(
        os.path.join(OUTPUT_DIR, 'model4_predictions.csv'), index=False
    )

    all_quant  = np.hstack([y_q10, y_q50, y_q90])
    quant_cols = (
        [c+'_q10' for c in TARGET_COLS] +
        [c+'_q50' for c in TARGET_COLS] +
        [c+'_q90' for c in TARGET_COLS]
    )
    pd.DataFrame(all_quant, columns=quant_cols).to_csv(
        os.path.join(OUTPUT_DIR, 'model4_quantile_predictions.csv'),
        index=False
    )

    torch.save({
        'model_state_dict': best_state,
        'model_config': {
            'n_features':  N_FEATURES,
            'd_model':     best_config['d_model'],
            'n_heads':     best_config['n_heads'],
            'n_layers':    best_config['n_layers'],
            'd_ff':        best_config['d_ff'],
            'dropout':     best_config['dropout'],
            'output_dim':  4,
            'n_quantiles': len(QUANTILES),
        },
        'feat_cols':   FEAT_COLS,
        'target_cols': TARGET_COLS,
        'quantiles':   QUANTILES,
    }, os.path.join(OUTPUT_DIR, 'model4_itransformer.pt'))

    summary = {
        'model':               'iTransformer_HyperSearch',
        'paper':               'Liu et al., ICLR 2024',
        'total_params':        total_params,
        'train_time_sec':      round(train_time, 2),
        'final_best_val_loss': round(best_val_final, 4),
        'avg_MAE':             round(avg_mae,  4),
        'avg_RMSE':            round(avg_rmse, 4),
        'avg_R2':              round(avg_r2,   4),
        'per_target':          per_target,
        'best_config':         best_config,
        'quantile_used':       'P90',
        'search_iterations':   N_SEARCH_ITER,
        'val_fraction':        VAL_FRACTION,
        'train_losses':        train_losses,
        'val_losses':          val_losses,
        'search_leaderboard': [
            {
                'rank':      rank,
                'candidate': r['candidate'],
                'val_loss':  r['val_loss'],
                'n_params':  r['n_params'],
                'config':    r['config'],
                'time_sec':  r['time_sec'],
            }
            for rank, r in enumerate(valid_results, 1)
        ],
    }

    with open(
        os.path.join(OUTPUT_DIR, 'model4_summary.json'), 'w'
    ) as f:
        json.dump(summary, f, indent=2)

    # Clean up progress file
    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)
        print("\nSearch progress file removed (run complete).")

    print(f"\nSaved to {OUTPUT_DIR}:")
    print(f"  model4_predictions.csv")
    print(f"  model4_quantile_predictions.csv")
    print(f"  model4_itransformer.pt")
    print(f"  model4_summary.json")
    print(f"\niTransformer complete.")

Using device: cuda
  GPU : NVIDIA GeForce RTX 4070 Ti SUPER
  VRAM: 17.2 GB

Data directory   : C:\Users\user1\Downloads\iTransformer
Output directory : C:\Users\user1\Downloads\iTransformer\model_outputs

Loading data...
  X_train : (201984, 126)
  X_test  : (135120, 126)
  Features: 123
  Dataset loaded successfully.

Creating time-ordered validation split (20%)...
  Train: 161,588   Val: 40,396

No previous progress found — starting fresh.

HYPERPARAMETER SEARCH  (8 candidates)
  Batch size    : 2048
  Search epochs : 35   Patience : 4

Candidate 01/8
  d_model=128  heads=4  layers=2  d_ff=512  drop=0.2  lr=1e-03  wd=1e-04  params=421,964
  Val loss : 0.0411  (1971s)
  Progress saved → model4_search_progress.json

Candidate 02/8
  d_model=64  heads=4  layers=4  d_ff=128  drop=0.1  lr=5e-04  wd=1e-05  params=144,556
  Val loss : 0.3212  (528s)
  Progress saved → model4_search_progress.json

Candidate 03/8
  d_model=64  heads=4  layers=4  d_ff=256  drop=0.1  lr=1e-04  wd=1e-05  params

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA used by PyTorch:", torch.version.cuda)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA used by PyTorch: 12.8
